# AirfRANS GNN Surrogate — Colab GPU setup

This notebook is infrastructure only: get the repo, the dependencies, and the
dataset onto a real GPU. Full-resolution mesh graphs (~180k nodes / ~720k edges
per case) hung the local dev machine even 5 at a time (CPU-only, 4GB GPU) — that's
exactly why real training doesn't happen locally. All real logic still lives in
`src/`; this notebook just proves the environment works, it doesn't contain
training logic itself.

Runtime > Change runtime type > GPU, before running anything below.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the repo

Colab VMs reset every session, so the repo needs to come from somewhere durable.
Push this repo to GitHub first, then replace the URL below. (Alternative: mount
Google Drive and keep the repo there instead — swap this cell for a Drive mount
if you'd rather not use GitHub.)

In [ ]:
REPO_URL = "https://github.com/Revanthkr1/airfrans-gnn-surrogate.git"

!git clone $REPO_URL repo
%cd repo

## 2. Install dependencies

Colab ships torch with CUDA already installed — don't reinstall it. `torch_geometric`
installs as pure Python (no `torch-scatter`/`torch-sparse` needed; our model only
uses `torch_geometric.utils.scatter`, confirmed working locally).

In [ ]:
!pip install -q torch_geometric lightning airfrans pyvista

## 3. Get the dataset

Same guard as the local scripts: skip the download if it's already there (the
raw `af.dataset.download()` call has no such check itself — it was the source of
a wasted repeat multi-GB download locally). If you've already got the dataset in
Drive from a previous session, mount Drive instead and point `DATA_ROOT` there to
skip re-downloading entirely.

In [ ]:
import os
import airfrans as af

DATA_ROOT = "data"
dataset_root = os.path.join(DATA_ROOT, "Dataset")

if not os.path.isdir(dataset_root) or not os.listdir(dataset_root):
    af.dataset.download(root=DATA_ROOT, file_name="Dataset", unzip=True, OpenFOAM=False)

print("manifest present:", os.path.isfile(os.path.join(dataset_root, "manifest.json")))

## 4. Smoke test: one full-resolution case on the real GPU

Same forward/backward timing check done locally on CPU (11s/case there) — here
it should be dramatically faster, and use the *real* mesh graph
(`build_graph`), not the local k-NN subsample workaround
(`build_subsampled_graph`, dev-machine-only, see `src/graph.py`).

In [ ]:
import time
import torch

from src.data import split_names
from src.dataset import PyGAirfRANSDataset
from src.model import MeshGraphNet

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

train_names = split_names(dataset_root, task="full", train=True)[:1]
ds = PyGAirfRANSDataset(dataset_root, train_names)  # no stats yet -- just a smoke test
data = ds[0].to(device)

model = MeshGraphNet().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

t0 = time.time()
pred = model(data.x, data.edge_index, data.edge_attr)
loss = torch.nn.functional.mse_loss(pred, data.y)
loss.backward()
opt.step()
print(f"nodes={data.x.shape[0]}, edges={data.edge_index.shape[1]}, "
      f"forward+backward+step={time.time()-t0:.2f}s, loss={loss.item():.4f}")

## 5. Preprocess: cache real mesh graphs to disk

One-time cost per case (VTU parsing dominates, not GPU-bound). Skips any case
already cached, so it's safe to re-run/resume. For 800 training cases this is
a genuinely long job -- consider saving `data/cache/` to Drive afterward so it
survives a Colab session reset.

In [ ]:
from src.data import split_names
from src.preprocess import preprocess_split

CACHE_DIR = os.path.join(DATA_ROOT, "cache", "full")
train_names = split_names(dataset_root, task="full", train=True)

preprocess_split(dataset_root, train_names, CACHE_DIR)
print(f"cached {len(train_names)} training cases to {CACHE_DIR}")

## 6. Train

Trains on the 800 official training cases minus a validation holdout (`n_val`),
reporting relative L2 per field (pressure, vx, vy, nu_t separately -- never one
blended number, see CLAUDE.md) on validation each epoch. The official 200-case
test split is deliberately untouched here -- it's reserved for week 5's lift/drag
evaluation, not used for training-time monitoring.

`batch_size` controls how many full graphs get combined per step -- tune it up
from here depending on how much GPU memory is actually available.

In [ ]:
from src.train import main as train_main

train_main(
    dataset_root=dataset_root,
    cache_dir=CACHE_DIR,
    stats_path=os.path.join(DATA_ROOT, "norm_stats.npz"),
    checkpoint_path=os.path.join(DATA_ROOT, "meshgraphnet.ckpt"),
    max_epochs=100,
    batch_size=4,
    n_val=80,
)